# Round 3 notebook 04 — hidden-pattern opportunity map

        This notebook is the synthesis layer: repeated timestamp checks, hidden-bot evidence,
        payoff distances, and candidate strategy hypotheses to test later.

        It does not implement a strategy.

In [1]:
import os
os.environ.setdefault("MPLCONFIGDIR", "/tmp/matplotlib")

from pathlib import Path
import math
import numpy as np
import pandas as pd
from IPython.display import display, Markdown

DATA = Path("data/ROUND3")
DAYS = [0, 1, 2]

def load_round3():
    prices = []
    trades = []
    for day in DAYS:
        p = pd.read_csv(DATA / f"prices_round_3_day_{day}.csv", sep=";")
        t = pd.read_csv(DATA / f"trades_round_3_day_{day}.csv", sep=";")
        p["day"] = day
        t["day"] = day
        prices.append(p)
        trades.append(t)
    prices = pd.concat(prices, ignore_index=True)
    trades = pd.concat(trades, ignore_index=True)
    for col in prices.columns:
        if col.startswith(("bid_", "ask_")) or col == "mid_price":
            prices[col] = pd.to_numeric(prices[col], errors="coerce")
    trades["price"] = pd.to_numeric(trades["price"], errors="coerce")
    trades["quantity"] = pd.to_numeric(trades["quantity"], errors="coerce")
    return prices, trades

def add_book_features(prices):
    p = prices.copy()
    p["best_bid"] = p["bid_price_1"]
    p["best_ask"] = p["ask_price_1"]
    p["spread"] = p["best_ask"] - p["best_bid"]
    bid_cols = ["bid_price_1", "bid_price_2", "bid_price_3"]
    ask_cols = ["ask_price_1", "ask_price_2", "ask_price_3"]
    p["wall_bid"] = p[bid_cols].min(axis=1)
    p["wall_ask"] = p[ask_cols].max(axis=1)
    p["wall_mid"] = (p["wall_bid"] + p["wall_ask"]) / 2
    p["wall_spread"] = p["wall_ask"] - p["wall_bid"]
    p["n_bid_levels"] = p[bid_cols].notna().sum(axis=1)
    p["n_ask_levels"] = p[ask_cols].notna().sum(axis=1)
    p["mid_minus_wall"] = p["mid_price"] - p["wall_mid"]
    return p

def add_future_mid(prices, horizons=(1, 5, 10, 25, 50, 100, 200, 500, 1000, 2000, 5000)):
    p = prices.sort_values(["product", "day", "timestamp"]).copy()
    for h in horizons:
        p[f"fut_mid_{h}"] = p.groupby(["product", "day"])["mid_price"].shift(-h)
        p[f"ret_{h}"] = p[f"fut_mid_{h}"] - p["mid_price"]
    return p

def attach_trades(prices, trades):
    t = trades.merge(
        prices,
        left_on=["day", "timestamp", "symbol"],
        right_on=["day", "timestamp", "product"],
        how="left",
    )
    t["side"] = np.where(
        t["price"] == t["ask_price_1"],
        "buy_agg",
        np.where(t["price"] == t["bid_price_1"], "sell_agg", "other"),
    )
    return t

def maker_markout_table(trades_with_book, horizons=(1, 10, 50, 100, 200, 500, 1000, 2000, 5000)):
    t = trades_with_book.copy()
    for h in horizons:
        # PnL for a passive maker who sold to an aggressive buyer, or bought from an aggressive seller.
        t[f"mk_{h}"] = np.where(
            t["side"] == "buy_agg",
            t["price"] - t[f"fut_mid_{h}"],
            np.where(t["side"] == "sell_agg", t[f"fut_mid_{h}"] - t["price"], np.nan),
        )
    agg = {
        "n": ("price", "size"),
        "avg_qty": ("quantity", "mean"),
        "avg_price": ("price", "mean"),
    }
    for h in horizons:
        agg[f"mk_{h}"] = (f"mk_{h}", "mean")
    return t.groupby(["symbol", "side"]).agg(**agg).round(3).reset_index()

def slow_ema_payoff(prices, span=1000, horizons=(10, 25, 50, 100, 200, 500, 1000, 2000)):
    rows = []
    p = prices.sort_values(["product", "day", "timestamp"]).copy()
    alpha = 2 / (span + 1)
    p["ema"] = p.groupby(["product", "day"])["mid_price"].transform(
        lambda s: s.ewm(alpha=alpha, adjust=False).mean()
    )
    p["dev"] = p["mid_price"] - p["ema"]
    for product, g in p.groupby("product"):
        if g["mid_price"].std() == 0:
            rows.append({"product": product, "dev_q10": 0, "dev_q90": 0, "best_H": None, "edge_ticks": 0})
            continue
        lo, hi = g["dev"].quantile(0.10), g["dev"].quantile(0.90)
        low = g[g["dev"] <= lo]
        high = g[g["dev"] >= hi]
        best = None
        for h in horizons:
            low_ret = low.groupby("day")["mid_price"].shift(-h) if False else low[f"ret_{h}"]
            high_ret = high[f"ret_{h}"]
            edge = (low_ret.mean() - high_ret.mean()) / 2
            rec = (h, edge, low_ret.mean(), high_ret.mean())
            if best is None or edge > best[1]:
                best = rec
        rows.append({
            "product": product,
            "dev_q10": lo,
            "dev_q90": hi,
            "best_H": best[0],
            "edge_ticks": best[1],
            "low_future_move": best[2],
            "high_future_move": best[3],
        })
    return pd.DataFrame(rows).sort_values("edge_ticks", ascending=False).round(3)

In [2]:
prices_raw, trades_raw = load_round3()
prices = add_future_mid(add_book_features(prices_raw))
trades = attach_trades(prices, trades_raw)

display(Markdown("## Repeated timestamp audit"))
rep_rows = []
for (sym, ts), g in trades.groupby(["symbol", "timestamp"]):
    if g["day"].nunique() >= 2:
        rep_rows.append({
            "symbol": sym,
            "timestamp": ts,
            "days_seen": g["day"].nunique(),
            "n": len(g),
            "sides": "".join(g.sort_values("day")["side"].str[0].fillna("?").tolist()),
            "qtys": tuple(g.sort_values("day")["quantity"].astype(int).tolist()),
        })
rep = pd.DataFrame(rep_rows)
if len(rep):
    display(rep.groupby("symbol").agg(repeated_timestamps=("timestamp", "nunique"), rows=("n", "sum")).sort_values("repeated_timestamps", ascending=False))
    display(rep[rep["symbol"].isin(["HYDROGEL_PACK", "VELVETFRUIT_EXTRACT"])].head(30))
else:
    display(Markdown("No repeated timestamp trades across days."))

display(Markdown("## Consolidated payoff-distance table"))
slow = slow_ema_payoff(prices, span=1000)
mk = maker_markout_table(trades)
mk_pivot = mk[mk["n"] >= 5].pivot_table(index="symbol", columns="side", values=["mk_100", "mk_500", "mk_1000"], aggfunc="first")
display(slow)
display(mk_pivot.round(3))

display(Markdown("## Opportunity map"))
opps = pd.DataFrame([
    {
        "edge_family": "HYDROGEL passive maker/recycler",
        "evidence": "Two quote layers plus balanced L1 takers; maker markout positive through ~1000 ticks.",
        "payoff_distance": "50-1000 ticks, then decays",
        "risk": "Backtester none undercounts passive fills; inventory carry beyond payoff horizon leaks.",
        "next_test": "Isolation backtest/website log: quote at/inside L1, flatten at wall/EMA fair, cap inventory age."
    },
    {
        "edge_family": "HYDROGEL fair correction",
        "evidence": "Wall/mid drifts around 9990, not fixed 10000; EMA wall fair outperformed fixed fair in prior audit.",
        "payoff_distance": "Hundreds to 2000 ticks for slow deviations",
        "risk": "Too-fast EMA chases noise; too-slow anchor carries regime drift.",
        "next_test": "Keep alpha in plateau 0.005-0.02; measure fill-quality markout not just PnL."
    },
    {
        "edge_family": "David ultra-slow voucher taker MR",
        "evidence": "All active vouchers mean-revert to slow anchor; v18 PnL dominated by 4500-5300.",
        "payoff_distance": "~1000 ticks on EMA1000 decile test",
        "risk": "Live TTE/surface level shift; large inventory pinned at limits.",
        "next_test": "Per-strike edge/size sweep with inventory-age limiter."
    },
    {
        "edge_family": "OTM voucher persistent seller",
        "evidence": "VEV_5300-5500 prints are 98-100% sell-aggressive into bid.",
        "payoff_distance": "Small positive maker markout, mostly 1-1000 ticks",
        "risk": "Spread=1 leaves little room; VEV_5500 fires rarely in none mode.",
        "next_test": "Asymmetric long-only bid accumulation for 5400/5500, no symmetric selling unless rich."
    },
    {
        "edge_family": "Pinned far-OTM vouchers",
        "evidence": "Trade prints at 0 show apparent +0.5 mid markout, but prior bid-0 backtest produced 0 realized EV.",
        "payoff_distance": "None tradable",
        "risk": "Classic visualization/mark-to-mid trap.",
        "next_test": "Do not trade unless visible ask at 0 appears or rules change."
    },
])
display(opps)

## Repeated timestamp audit

,repeated_timestamps,rows
symbol,,
VELVETFRUIT_EXTRACT,60,122
HYDROGEL_PACK,24,48
VEV_4000,7,14
VEV_5400,5,10
VEV_5500,5,10
VEV_6000,5,10
VEV_6500,5,10
VEV_5300,2,4


,symbol,timestamp,days_seen,n,sides,qtys
0,HYDROGEL_PACK,15700,2,2,sb,"(2, 2)"
1,HYDROGEL_PACK,164600,2,2,ss,"(3, 6)"
2,HYDROGEL_PACK,169000,2,2,sb,"(3, 2)"
3,HYDROGEL_PACK,178100,2,2,ss,"(2, 3)"
4,HYDROGEL_PACK,185800,2,2,sb,"(6, 4)"
5,HYDROGEL_PACK,193400,2,2,bb,"(4, 4)"
6,HYDROGEL_PACK,206200,2,2,bb,"(5, 3)"
7,HYDROGEL_PACK,206700,2,2,ss,"(5, 5)"
8,HYDROGEL_PACK,221200,2,2,bb,"(5, 6)"
9,HYDROGEL_PACK,273500,2,2,bb,"(4, 6)"


## Consolidated payoff-distance table

,product,dev_q10,dev_q90,best_H,edge_ticks,low_future_move,high_future_move
0,HYDROGEL_PACK,-25.577,26.477,2000.0,32.039,35.430,-28.648
2,VEV_4000,-13.900,14.648,1000.0,20.881,23.099,-18.664
3,VEV_4500,-13.846,14.633,1000.0,20.879,23.058,-18.700
1,VELVETFRUIT_EXTRACT,-13.886,14.654,1000.0,20.770,22.864,-18.676
4,VEV_5000,-12.984,13.598,1000.0,19.531,21.280,-17.782
5,VEV_5100,-11.534,11.997,1000.0,16.831,17.917,-15.745
6,VEV_5200,-8.717,8.890,1000.0,12.638,12.839,-12.436
7,VEV_5300,-5.499,5.599,1000.0,8.101,7.792,-8.410
8,VEV_5400,-2.684,2.536,1000.0,3.497,2.607,-4.387
9,VEV_5500,-1.213,1.146,1000.0,1.595,1.174,-2.017


mk_100          mk_1000           mk_500         
side                buy_agg sell_agg buy_agg sell_agg buy_agg sell_agg
symbol                                                                
HYDROGEL_PACK         7.274    8.282   2.909   12.336   4.427    8.825
VELVETFRUIT_EXTRACT   1.353    2.254   0.082    4.222   0.977    2.163
VEV_4000             10.207    9.722   9.814   10.482  10.384   10.581
VEV_5200                NaN    1.000     NaN    8.800     NaN    3.906
VEV_5300                NaN    0.829     NaN    2.722     NaN    0.942
VEV_5400                NaN    0.489     NaN    0.745     NaN    0.616
VEV_5500                NaN    0.519     NaN    0.612     NaN    0.525
VEV_6000                NaN    0.500     NaN    0.500     NaN    0.500
VEV_6500                NaN    0.500     NaN    0.500     NaN    0.500

## Opportunity map

,edge_family,evidence,payoff_distance,risk,next_test
0,HYDROGEL passive maker/recycler,Two quote layers plus balanced L1 takers; make...,"50-1000 ticks, then decays",Backtester none undercounts passive fills; inv...,Isolation backtest/website log: quote at/insid...
1,HYDROGEL fair correction,"Wall/mid drifts around 9990, not fixed 10000; ...",Hundreds to 2000 ticks for slow deviations,Too-fast EMA chases noise; too-slow anchor car...,Keep alpha in plateau 0.005-0.02; measure fill...
2,David ultra-slow voucher taker MR,All active vouchers mean-revert to slow anchor...,~1000 ticks on EMA1000 decile test,Live TTE/surface level shift; large inventory ...,Per-strike edge/size sweep with inventory-age ...
3,OTM voucher persistent seller,VEV_5300-5500 prints are 98-100% sell-aggressi...,"Small positive maker markout, mostly 1-1000 ticks",Spread=1 leaves little room; VEV_5500 fires ra...,Asymmetric long-only bid accumulation for 5400...
4,Pinned far-OTM vouchers,Trade prints at 0 show apparent +0.5 mid marko...,None tradable,Classic visualization/mark-to-mid trap.,Do not trade unless visible ask at 0 appears o...


## Synthesis

        The clearest hidden structure is not a complicated predictive model. It is the ecology of scripted bots:

        - HYDROGEL has layered quote bots and balanced taker bots. The edge is spread capture plus bounded-horizon recycling.
        - The voucher surface has a persistent seller in OTM strikes and a slow-anchor reversion process in active strikes.
        - Repeated timestamp reuse exists in small amounts but is not strong enough to be the primary edge in R3.
        - The next strategy should be built from interpretable bot roles and payoff horizons, not from stacked feature models.